# CH4Net v8 · Methane Portfolio Risk Dashboard
## Satellite-Derived Carbon Liability Analysis — EU Industrial Emitters
---
**Model**: CH4Net v2 (13.5M param U-Net, Sentinel-2 L1C input, 100×100 patch tiling)  
**Coverage**: 6 sites × up to 4 acquisition dates (Summer 2024)  
**Pipeline**: Copernicus S2 L1C → CH4Net inference → S/C ratio → CFAR detection → IME/CEMF quantification → EU ETS carbon liability  
**API**: `POST /portfolio-risk`, `GET /site-risk/{site_name}`


In [ ]:
import json, math, sys, os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

# Add project root so we can import RiskModel directly
sys.path.insert(0, str(Path.cwd()))

# Suppress torch/rasterio imports if not available
import unittest.mock as mock
for mod in ['torch', 'rasterio', 'rasterio.transform', 'rasterio.crs']:
    try:
        __import__(mod)
    except ImportError:
        sys.modules[mod] = mock.MagicMock()

from src.api.risk_model import RiskModel, SITE_OPERATOR_MAP, TICKER_SITES

DARK = '#0f1117'
plt.rcParams.update({
    'figure.facecolor': DARK, 'axes.facecolor': DARK,
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'text.color': '#eee', 'xtick.color': '#aaa', 'ytick.color': '#aaa',
    'grid.color': '#2a2a2a', 'legend.facecolor': '#1c1c2e',
    'legend.edgecolor': '#555', 'font.family': 'DejaVu Sans', 'figure.dpi': 120,
})

RISK_COLORS = {
    'HIGH': '#ff4444',
    'HIGH_DUAL_SENSOR': '#ff6600',
    'MEDIUM': '#ffaa00',
    'LOW': '#88cc44',
    'UNDETECTED': '#4488cc',
    'NO_DATA': '#888',
}

print('Imports OK')


## 1. Load Validation Data

In [ ]:
with open('results_analysis/multidate_validation.json') as f:
    md_raw = json.load(f)
with open('results_analysis/quantification.json') as f:
    quant_raw = json.load(f)
quant_by_site = {r['site']: r for r in quant_raw}

BAD_SCENE_MEAN = 0.4257
rows = []
for site, data in md_raw.items():
    op_info = SITE_OPERATOR_MAP.get(site, {})
    for date_str, r in data.get('dates', {}).items():
        sc = r.get('sc_ratio')
        sm = r.get('site_mean')
        is_bad = sm is not None and abs(sm - BAD_SCENE_MEAN) < 0.01
        rows.append({
            'site': site,
            'operator': op_info.get('operator', 'Unknown'),
            'ticker': op_info.get('ticker'),
            'date': pd.to_datetime(date_str, format='%Y%m%d'),
            'sc_ratio': sc,
            'site_mean': sm,
            'detected': bool(r.get('classic_detect', False)),
            'is_bad_scene': is_bad,
            'valid': (sc is not None) and (not is_bad),
        })

df = pd.DataFrame(rows).sort_values(['site', 'date'])
print(f'Total observations: {len(df)}  |  Valid: {df.valid.sum()}  |  Detected: {df.detected.sum()}')
print(df[['site', 'date', 'sc_ratio', 'is_bad_scene', 'detected', 'valid']].to_string(index=False))


## 2. Risk Model — Per-Site Carbon Liability

In [ ]:
model = RiskModel()

all_sites = sorted(SITE_OPERATOR_MAP.keys())
risk_rows = []
for site in all_sites:
    score = model.site_risk(site)
    risk_rows.append(score)

risk_df = pd.DataFrame(risk_rows)
risk_df = risk_df.sort_values('annual_tCO2e', ascending=False, na_position='last')

display_cols = [
    'site', 'operator', 'ticker', 'p_detect', 'n_detections', 'n_valid_dates',
    'mean_sc_detected', 'flow_rate_kgh', 'annual_tCO2e', 'carbon_liability_eur', 'risk_tier'
]
print(risk_df[display_cols].to_string(index=False))


## 3. Portfolio Risk API Demo
Live call against the `RiskModel` (same logic as `POST /portfolio-risk`):
```
POST http://localhost:8000/portfolio-risk
{"tickers": ["RWE.DE", "PGE.WA", "SHEL.L"], "lookback_days": 180}
```


In [ ]:
PORTFOLIO = ['RWE.DE', 'PGE.WA', 'SHEL.L', 'UN01.DE']

port = model.portfolio_risk(PORTFOLIO, lookback_days=180)

print('Portfolio Carbon Liability Summary')
print('=' * 60)
print(f'Total annual CH4 (CO2e):  {port["total_annual_tCO2e"]:>12,.0f} t/yr')
print(f'Total carbon liability:   E{port["total_carbon_eur"]:>11,.0f} / yr')
print(f'Data coverage:             {port["data_coverage_pct"]:.0f}% of submitted tickers')
print(f'Unmatched tickers:         {port["unmatched_tickers"]}')
print()

for ticker, td in port['per_ticker'].items():
    print('-' * 50)
    print(f'  {ticker}  [{td["risk_tier"]}]')
    print(f'  Sites:          {chr(44).join(td["sites"])}')
    print(f'  Annual tCO2e:   {td["annual_tCO2e"]:>10,.0f}  '
          f'[{td["annual_tCO2e_lo_90"]:,.0f} - {td["annual_tCO2e_hi_90"]:,.0f}] 90pct CI')
    print(f'  Carbon EUR/yr:  E{td["carbon_liability_eur"]:>10,.0f}  '
          f'[E{td["carbon_eur_lo_90"]:,.0f} - E{td["carbon_eur_hi_90"]:,.0f}]')


## 4. S/C Ratio Time Series
Each point is one Sentinel-2 acquisition. S/C = site_mean / ctrl_mean.  
Detection threshold: **S/C > 1.15** (CFAR, CV-adaptive). Bad scenes (degenerate model output where site_mean = 0.4257) are marked separately.


In [ ]:
sites_to_plot = ['weisweiler', 'belchatow', 'neurath', 'boxberg', 'lippendorf', 'niederaussem']
fig, axes = plt.subplots(2, 3, figsize=(16, 9), facecolor=DARK)
fig.suptitle('CH4Net S/C Ratio by Site and Date  (Summer 2024 Sentinel-2)',
             color='white', fontsize=15, fontweight='bold', y=0.98)

for ax, site in zip(axes.flat, sites_to_plot):
    site_df = df[df.site == site].copy()
    op_info = SITE_OPERATOR_MAP.get(site, {})
    score = model.site_risk(site)
    tier = score['risk_tier']
    tier_col = RISK_COLORS.get(tier, '#888')

    valid_df = site_df[site_df.valid]
    bad_df   = site_df[site_df.is_bad_scene]

    ax.axhline(y=1.15, color='#ffaa00', linewidth=0.8, linestyle='--', alpha=0.6)

    for _, row in valid_df.iterrows():
        color  = '#ff4444' if row.detected else '#4488cc'
        marker = 'D' if row.detected else 'o'
        ax.scatter(row.date, row.sc_ratio, c=color, marker=marker, s=100, zorder=5, alpha=0.9)

    for _, row in bad_df.iterrows():
        ax.scatter(row.date, 0.98, c='#555', marker='x', s=80, zorder=4, alpha=0.7)
        ax.text(row.date, 1.04, 'bad\nscene', color='#666', fontsize=5.5, ha='center')

    if len(valid_df) > 1:
        ax.plot(valid_df.date, valid_df.sc_ratio, color='#336699', linewidth=0.8, alpha=0.5)

    p = score['p_detect']
    lo = score['p_detect_lo_90']
    hi = score['p_detect_hi_90']
    if p is not None:
        p_str = f'p={p:.0%} [{lo:.0%}-{hi:.0%}]'
    else:
        p_str = 'p=N/A'

    op_name = op_info.get('operator', '?')
    ax.set_title(f'{site.upper()}\n{op_name}  {p_str}',
                 color=tier_col, fontsize=8.5, pad=4)
    ax.set_yscale('log')
    valid_sc = valid_df.sc_ratio.dropna()
    ymax = float(valid_sc.max()) * 1.2 if len(valid_sc) > 0 else 2.0
    ax.set_ylim(bottom=0.005, top=max(ymax, 1.5))
    ax.set_ylabel('S/C ratio (log)', fontsize=7, color='#aaa')
    ax.tick_params(axis='x', labelrotation=30, labelsize=7)
    ax.tick_params(axis='y', labelsize=7)
    ax.spines[:].set_color('#333')
    ax.grid(True, which='both', alpha=0.2)
    ax.text(0.98, 0.97, tier, transform=ax.transAxes, ha='right', va='top',
            color=tier_col, fontsize=8, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='#1a1a2e', edgecolor=tier_col, alpha=0.8))

det_p   = mpatches.Patch(color='#ff4444', label='Detected (S/C > 1.15)')
miss_p  = mpatches.Patch(color='#4488cc', label='Non-detection')
bad_p   = mpatches.Patch(color='#555',    label='Bad scene (degenerate output)')
thresh  = plt.Line2D([0], [0], color='#ffaa00', linestyle='--', label='CFAR threshold 1.15')
fig.legend(handles=[det_p, miss_p, bad_p, thresh],
           loc='lower center', ncol=4, fontsize=8,
           facecolor='#1c1c2e', edgecolor='#555', labelcolor='white',
           bbox_to_anchor=(0.5, 0.0))

plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig('results_analysis/sc_ratio_timeseries.png', dpi=150, bbox_inches='tight', facecolor=DARK)
plt.show()
print('Saved: results_analysis/sc_ratio_timeseries.png')


## 5. Carbon Liability by Site (EU ETS Pricing)
Expected annual methane emission costs at **65 EUR/tCO2e** (EU ETS).  
Error bars = 90% CI (Wilson on p_detect x +/-50% flow-rate uncertainty).


In [ ]:
liability_df = risk_df[risk_df.carbon_liability_eur.notna()].copy()
liability_df = liability_df.sort_values('carbon_liability_eur', ascending=True)

fig, ax = plt.subplots(figsize=(12, 6), facecolor=DARK)
ax.set_facecolor(DARK)

y_pos  = range(len(liability_df))
colors = [RISK_COLORS.get(t, '#888') for t in liability_df.risk_tier]

lo_err = [(liability_df.iloc[i].carbon_liability_eur - liability_df.iloc[i].carbon_eur_lo_90) / 1e6
          for i in range(len(liability_df))]
hi_err = [(liability_df.iloc[i].carbon_eur_hi_90 - liability_df.iloc[i].carbon_liability_eur) / 1e6
          for i in range(len(liability_df))]

ax.barh(
    list(y_pos),
    liability_df.carbon_liability_eur.values / 1e6,
    xerr=[lo_err, hi_err],
    color=colors, alpha=0.85, height=0.6,
    error_kw=dict(elinewidth=1.2, capsize=4, ecolor='#888', capthick=1.2),
)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(
    [f"{r.site}\n{r.operator or ''}" for _, r in liability_df.iterrows()],
    fontsize=9
)

for i, (_, row) in enumerate(liability_df.iterrows()):
    val = row.carbon_liability_eur
    if val and val > 0:
        hi = row.carbon_eur_hi_90 or val
        ax.text(hi / 1e6 + 0.05, i, f'E{val/1e6:.1f}M', va='center', fontsize=8, color='#ccc')

ax.set_xlabel('Annual Carbon Liability (EUR million / yr)', color='#ccc', fontsize=10)
ax.set_title(
    'Estimated CH4 Carbon Liability  EU ETS @ 65 EUR/tCO2e  (90% CI)\n'
    'CH4Net v8  Summer 2024 Sentinel-2  |  Flow: CEMF/IME or S/C proxy',
    color='white', fontsize=12, pad=10
)

legend_patches = [mpatches.Patch(color=c, label=t) for t, c in RISK_COLORS.items() if t != 'NO_DATA']
ax.legend(handles=legend_patches, title='Risk Tier', title_fontsize=8, fontsize=8,
          loc='lower right', facecolor='#1c1c2e', edgecolor='#555', labelcolor='white')
ax.spines[:].set_color('#333')
ax.tick_params(colors='#aaa')
ax.grid(axis='x', alpha=0.2)
ax.set_xlim(left=0)

plt.tight_layout()
plt.savefig('results_analysis/carbon_liability_sites.png', dpi=150, bbox_inches='tight', facecolor=DARK)
plt.show()
print('Saved: results_analysis/carbon_liability_sites.png')


## 6. Portfolio Carbon Exposure by Ticker

In [ ]:
port = model.portfolio_risk(['RWE.DE', 'PGE.WA', 'SHEL.L', 'UN01.DE'])

tickers = list(port['per_ticker'].keys())
vals    = [port['per_ticker'][t]['carbon_liability_eur'] / 1e6 for t in tickers]
lo_err  = [(port['per_ticker'][t]['carbon_liability_eur'] - port['per_ticker'][t]['carbon_eur_lo_90']) / 1e6
           for t in tickers]
hi_err  = [(port['per_ticker'][t]['carbon_eur_hi_90'] - port['per_ticker'][t]['carbon_liability_eur']) / 1e6
           for t in tickers]
tiers   = [port['per_ticker'][t]['risk_tier'] for t in tickers]
clrs    = [RISK_COLORS.get(t, '#888') for t in tiers]

fig, ax = plt.subplots(figsize=(10, 5), facecolor=DARK)
ax.set_facecolor(DARK)

ax.bar(tickers, vals, yerr=[lo_err, hi_err], color=clrs, alpha=0.85,
       error_kw=dict(elinewidth=1.5, capsize=6, ecolor='#888', capthick=1.5))

max_val = max(vals) if vals else 1.0
for i, (t, v) in enumerate(zip(tickers, vals)):
    if v > 0:
        ax.text(i, v + hi_err[i] + max_val * 0.01, f'E{v:.1f}M', ha='center', fontsize=9, color='#eee')
    tier = tiers[i]
    ax.text(i, -max_val * 0.05, tier, ha='center', fontsize=7.5, color=RISK_COLORS.get(tier, '#888'))

total_eur = port['total_carbon_eur']
ets_price = port['ets_price_eur_tonne']
ax.set_ylabel('Annual EU ETS Carbon Liability (EUR million / yr)', color='#ccc', fontsize=10)
ax.set_title(
    f'Portfolio Methane Carbon Exposure  |  Total: E{total_eur/1e6:.1f}M/yr\n'
    f'CH4Net v8  S/C-proxy flow rates  EU ETS E{ets_price}/tCO2e  (90% CI error bars)',
    color='white', fontsize=12, pad=10
)
ax.spines[:].set_color('#333')
ax.tick_params(colors='#aaa', labelsize=11)
ax.grid(axis='y', alpha=0.2)
ax.set_ylim(bottom=-max_val * 0.12)

plt.tight_layout()
plt.savefig('results_analysis/portfolio_carbon_exposure.png', dpi=150,
            bbox_inches='tight', facecolor=DARK)
plt.show()
print(f'Saved: results_analysis/portfolio_carbon_exposure.png')
print(f'Total portfolio liability: E{total_eur/1e6:.1f}M/yr  ({port["data_coverage_pct"]:.0f}% coverage)')


## 7. EU Emission Site Map

In [ ]:
SITE_COORDS = {
    'weisweiler':     (50.837,  6.322),
    'belchatow':      (51.266, 19.315),
    'lippendorf':     (51.178, 12.378),
    'boxberg':        (51.413, 14.582),
    'neurath':        (51.040,  6.683),
    'niederaussem':   (50.997,  6.656),
    'rybnik':         (50.090, 18.529),
    'groningen':      (53.252,  6.682),
    'jaenschwalde':   (51.838, 14.621),
    'schwarze_pumpe': (51.579, 14.328),
    'turow':          (51.003, 14.988),
    'maasvlakte':     (51.950,  4.020),
}

MAP_BG = '#0a0e1a'
fig, ax = plt.subplots(figsize=(14, 8), facecolor=MAP_BG)
ax.set_facecolor(MAP_BG)
ax.set_xlim(-2, 25)
ax.set_ylim(47, 57)
ax.grid(True, color='#1a2035', linewidth=0.5, linestyle='--', alpha=0.7)
ax.set_xlabel('Longitude (deg E)', color='#aaa', fontsize=9)
ax.set_ylabel('Latitude (deg N)', color='#aaa', fontsize=9)

# Rough country outlines
germany = [(6.0,47.3),(6.8,48.0),(8.0,47.6),(9.5,47.5),(10.5,47.3),
           (13.0,47.7),(15.0,50.5),(14.8,51.0),(15.0,51.2),(14.5,53.0),
           (13.8,54.2),(11.0,54.0),(9.0,54.8),(8.5,55.0),(7.0,53.5),
           (6.0,51.5),(6.1,50.0),(6.3,49.5),(6.0,47.3)]
gx, gy = zip(*germany)
ax.plot(gx, gy, color='#2a3a5a', linewidth=1.2, alpha=0.8)

poland = [(14.5,53.0),(18.5,54.8),(22.0,54.3),(24.0,53.5),(23.5,52.0),
          (24.0,50.5),(22.0,49.0),(18.0,49.6),(15.0,50.5),(14.5,51.0),(14.5,53.0)]
px, py = zip(*poland)
ax.plot(px, py, color='#2a3a5a', linewidth=1.2, alpha=0.8)

nl = [(3.4,51.4),(4.2,51.5),(5.5,51.0),(6.1,50.8),(6.8,51.0),
      (7.2,53.3),(6.5,53.7),(5.0,53.5),(4.0,53.0),(3.4,51.4)]
nx, ny = zip(*nl)
ax.plot(nx, ny, color='#2a3a5a', linewidth=1.2, alpha=0.8)

for site, (lat, lon) in SITE_COORDS.items():
    score = model.site_risk(site)
    tier  = score['risk_tier']
    color = RISK_COLORS.get(tier, '#888')
    tco2  = score.get('annual_tCO2e') or 0
    size  = max(60, min(600, tco2 / 100)) if tco2 > 0 else 60
    ax.scatter(lon, lat, s=size, c=color, alpha=0.85, zorder=5,
               edgecolors='white', linewidths=0.6)
    tick  = score['ticker']
    p     = score['p_detect']
    p_str = f'{p:.0%}' if p is not None else 'N/A'
    label = site.replace('_', ' ').upper()
    ax.annotate(
        f'{label}\n{tick or "(private)"}  p={p_str}',
        xy=(lon, lat), xytext=(lon + 0.25, lat + 0.18),
        fontsize=6.5, color='#ddd',
        arrowprops=dict(arrowstyle='-', color='#555', lw=0.5),
        bbox=dict(boxstyle='round,pad=0.2', facecolor=MAP_BG, edgecolor='#333', alpha=0.8),
    )

legend_patches = [mpatches.Patch(color=c, label=t) for t, c in RISK_COLORS.items()]
ax.legend(handles=legend_patches, title='Risk Tier', title_fontsize=8, fontsize=8,
          loc='lower right', facecolor='#1c1c2e', edgecolor='#555', labelcolor='white')
ax.set_title(
    'CH4Net v8  European Methane Emitter Map\n'
    'Bubble size proportional to expected annual tCO2e  |  p = Wilson 90% CI centre',
    color='white', fontsize=12, pad=10
)
ax.tick_params(colors='#aaa', labelsize=8)
ax.spines[:].set_color('#333')
plt.tight_layout()
plt.savefig('results_analysis/eu_emitter_map.png', dpi=150, bbox_inches='tight', facecolor=MAP_BG)
plt.show()
print('Saved: results_analysis/eu_emitter_map.png')


## 8. Detection Statistics Summary Table

In [ ]:
summary_rows = []
for site in sorted(SITE_OPERATOR_MAP.keys()):
    score = model.site_risk(site)
    op = SITE_OPERATOR_MAP[site]

    p = score['p_detect']
    lo = score['p_detect_lo_90']
    hi = score['p_detect_hi_90']

    summary_rows.append({
        'Site':           site.replace('_', ' ').title(),
        'Operator':       score['operator'] or '-',
        'Ticker':         score['ticker'] or 'private',
        'n_dates':        score['n_valid_dates'],
        'n_detect':       score['n_detections'],
        'p_detect':       f'{p:.0%}' if p is not None else '-',
        'CI_90':          f'[{lo:.0%}-{hi:.0%}]' if p is not None else '-',
        'Mean S/C':       f'{score["mean_sc_detected"]:.1f}' if score['mean_sc_detected'] else '-',
        'Flow kg/hr':     f'{score["flow_rate_kgh"]:.0f}' if score['flow_rate_kgh'] else '-',
        'Ann tCO2e':      f'{score["annual_tCO2e"]:,.0f}' if score['annual_tCO2e'] else '-',
        'Carbon EUR/yr':  f'E{score["carbon_liability_eur"]:,.0f}' if score['carbon_liability_eur'] else '-',
        'Risk Tier':      score['risk_tier'],
    })

summary_table = pd.DataFrame(summary_rows)
print(summary_table.to_string(index=False))


## 9. Bi-Temporal Analysis (Terrain Artifact Suppression)
BT replaces raw S/C with delta-B12-based S/C to suppress seasonal terrain artifacts.  
Key finding: Lippendorf collapses **315 -> 0.195** under BT (summer SWIR vegetation artifact confirmed).


In [ ]:
with open('results_analysis/bitemporal_comparison.json') as f:
    bt_data = json.load(f)

header = f"{'Site':<16} {'Original S/C':>14} {'BT S/C':>10} {'Verdict':>28}"
print(header)
print('-' * 72)

for site, data in bt_data.items():
    orig = data.get('original', {})
    bt   = data.get('bitemporal', {})
    orig_sc = orig.get('sc_ratio')
    bt_sc   = bt.get('sc_ratio')

    if orig_sc is not None and bt_sc is not None:
        if bt_sc < 1.15:
            verdict = 'ARTIFACT (BT suppressed signal)'
        elif bt_sc / orig_sc > 0.5:
            verdict = 'Persistent (real emission)'
        else:
            verdict = 'Partial suppression'
        print(f'{site:<16} {orig_sc:>14.3f} {bt_sc:>10.3f}   {verdict}')
    else:
        print(f'{site:<16} {"N/A":>14} {"N/A":>10}   no data')

print()
print('Lippendorf: B12 seasonal delta = -22 DN -> vegetation SWIR artifact confirmed')
print('Belchatow: B12 delta = +3 DN -> nearly neutral -> BT suppression expected for industrial sites')


## 10. TROPOMI Cross-Validation *(pending)*
Run `validate_tropomi.py` to download S5P OFFL L2 XCH4 orbits and generate dual-sensor confirmation.

```bash
pip install netCDF4 --break-system-packages
caffeinate -i python validate_tropomi.py
```

Once `results_analysis/tropomi_validation.json` is generated, this section will render:
- TROPOMI delta-XCHx vs CH4Net S/C scatter per site
- Dual-sensor detection rate heatmap
- Upgraded risk tiers for dual-sensor-confirmed sites


In [ ]:
TROPOMI_PATH = Path('results_analysis/tropomi_validation.json')

if TROPOMI_PATH.exists():
    with open(TROPOMI_PATH) as f:
        trop = json.load(f)
    print(f'TROPOMI data available for {len(trop)} sites')

    scatter_rows = []
    for site, sdata in trop.items():
        for date, d in sdata.get('dates', {}).items():
            if 'error' not in d.get('tropomi', {}):
                scatter_rows.append({
                    'site': site,
                    'sc_ratio': d.get('sc_ratio'),
                    'dxch4_ppb': d.get('tropomi', {}).get('delta_xch4_ppb'),
                    's2_detect': d.get('s2_detect'),
                    'trop_detect': d.get('trop_detect'),
                })

    if scatter_rows:
        scat_df = pd.DataFrame(scatter_rows).dropna(subset=['sc_ratio', 'dxch4_ppb'])
        fig, ax = plt.subplots(figsize=(10, 7), facecolor=DARK)
        ax.set_facecolor(DARK)
        color_map = {(True, True):'#ff4444', (True,False):'#ff9900',
                     (False,True):'#4499ff', (False,False):'#666'}
        label_map = {
            (True,True):  'Dual-sensor confirm',
            (True,False): 'S2-only detect',
            (False,True): 'TROPOMI-only',
            (False,False):'Non-detection',
        }
        for (s2, td2), grp in scat_df.groupby(['s2_detect', 'trop_detect']):
            ax.scatter(grp.sc_ratio, grp.dxch4_ppb,
                      c=color_map.get((s2,td2),'#666'), s=80, alpha=0.8,
                      label=label_map.get((s2,td2),''))
        ax.axhline(y=5.0, color='#4499ff', linewidth=0.8, linestyle='--', alpha=0.6, label='TROPOMI thresh 5 ppb')
        ax.axvline(x=1.15, color='#ffaa00', linewidth=0.8, linestyle='--', alpha=0.6, label='S/C thresh 1.15')
        ax.set_xscale('log')
        ax.set_xlabel('CH4Net S/C Ratio (log)', color='#ccc', fontsize=10)
        ax.set_ylabel('TROPOMI delta-XCHx (ppb)', color='#ccc', fontsize=10)
        ax.set_title('Dual-Sensor Cross-Validation: CH4Net S/C vs TROPOMI delta-XCHx',
                     color='white', fontsize=12)
        ax.legend(fontsize=8, facecolor='#1c1c2e', edgecolor='#555', labelcolor='white')
        ax.spines[:].set_color('#333')
        ax.grid(alpha=0.2)
        plt.tight_layout()
        plt.show()
else:
    print('tropomi_validation.json not found. Run validate_tropomi.py to generate TROPOMI data.')
    print()
    print('Command:')
    print('  pip install netCDF4 --break-system-packages')
    print('  caffeinate -i python validate_tropomi.py')
    print()
    print('Once run, this section shows:')
    print('  S/C vs delta-XCHx scatter with dual-sensor confirmation markers')
    print('  Detection rate heatmap by site x date')
    print('  Updated risk tiers (HIGH_DUAL_SENSOR for confirmed sites)')
